# 🔁 PLM Data Migration & Testing Simulation
### Simulated Migration of Product Data Between Systems
**Author:** Mausamkumari Thakur     
**Tools:** Python · Pandas · SQL Logic · Excel Reports  
**Entities:** Parts & BOM · Documents · ECO/ECR  
**Phases:** Data Extraction · Validation · Migration QA · Defect Reporting

---
## 📋 Project Scope
This notebook simulates a real-world PLM data migration scenario covering:
- Extraction and profiling of source PLM data
- Data integrity and business rule validation
- Structured test case execution (SDLC/UAT aligned)
- Defect logging and migration status reporting

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')

✅ Libraries loaded
   Pandas  : 2.2.2
   NumPy   : 2.0.2


---
## Load Source PLM Data

In [2]:
parts  = pd.read_csv('source_parts.csv')
bom    = pd.read_csv('source_bom.csv')
docs   = pd.read_csv('source_documents.csv')
eco    = pd.read_csv('source_eco.csv')

for name, df in [('Parts',parts),('BOM',bom),('Documents',docs),('ECO',eco)]:
    print(f'  {name:12s}: {df.shape[0]} rows × {df.shape[1]} cols')

  Parts       : 200 rows × 13 cols
  BOM         : 150 rows × 9 cols
  Documents   : 100 rows × 11 cols
  ECO         : 80 rows × 11 cols


---
## Data Profiling & Extraction Checks

In [3]:
print('=== PARTS PROFILE ===')
print(parts.dtypes)
print('\nMissing values:')
print(parts.isnull().sum())
print('\nStatus distribution:')
print(parts['Status'].value_counts())
print('\nMigration Status:')
print(parts['Migration_Status'].value_counts())

=== PARTS PROFILE ===
Part_ID              object
Part_Name            object
Category             object
Revision             object
Status               object
UOM                  object
Weight_KG           float64
Unit_Cost_INR       float64
Owner                object
Created_Date         object
Modified_Date        object
Source_System        object
Migration_Status     object
dtype: object

Missing values:
Part_ID             0
Part_Name           1
Category            0
Revision            1
Status              0
UOM                 0
Weight_KG           1
Unit_Cost_INR       0
Owner               0
Created_Date        0
Modified_Date       0
Source_System       0
Migration_Status    0
dtype: int64

Status distribution:
Status
Released          95
Obsolete          38
In Review         36
Draft             30
INVALID_STATUS     1
Name: count, dtype: int64

Migration Status:
Migration_Status
Completed      87
Pending        51
In Progress    45
Failed         17
Name: count, dty

In [4]:
print('=== BOM PROFILE ===')
print(bom.isnull().sum())
print('\nBOM Status:')
print(bom['BOM_Status'].value_counts())

print('\n=== DOCUMENTS PROFILE ===')
print(docs.isnull().sum())
print('\nDoc Types:')
print(docs['Doc_Type'].value_counts())

print('\n=== ECO PROFILE ===')
print(eco.isnull().sum())
print('\nECO Status:')
print(eco['Status'].value_counts())

=== BOM PROFILE ===
BOM_ID              0
Parent_Part_ID      1
Child_Part_ID       0
Quantity            0
BOM_Level           0
Effectivity_Date    0
BOM_Status          0
Source_System       0
Migration_Status    0
dtype: int64

BOM Status:
BOM_Status
Active      98
Inactive    33
Pending     19
Name: count, dtype: int64

=== DOCUMENTS PROFILE ===
Doc_ID              0
Doc_Name            1
Doc_Type            0
Related_Part_ID     0
Version             0
File_Format         0
File_Size_MB        0
Owner               0
Created_Date        0
Source_System       0
Migration_Status    0
dtype: int64

Doc Types:
Doc_Type
User Manual      26
Drawing          25
SOP              21
Test Report      19
Specification     9
Name: count, dtype: int64

=== ECO PROFILE ===
ECO_ID               0
ECO_Type             0
Title                0
Affected_Part_ID     0
Priority             1
Status               0
Initiated_By         0
Initiated_Date       0
Approved_By         21
Source_System    

---
##  — Data Validation Engine
Simulates SQL-based validation checks for data integrity, referential integrity, and business rules.

In [5]:
VALID_STATUSES    = {'Released','In Review','Obsolete','Draft'}
VALID_PRIORITIES  = {'Critical','High','Medium','Low'}
VALID_ECO_STS     = {'Open','In Review','Approved','Rejected','Closed'}
valid_part_ids    = set(parts['Part_ID'])

from datetime import datetime
issues = []

def log(entity, record_id, field, issue_type, detail, severity='High'):
    issues.append({'Entity':entity,'Record_ID':record_id,'Field':field,
                   'Issue_Type':issue_type,'Detail':detail,'Severity':severity,
                   'Detected_At':datetime.now().strftime('%Y-%m-%d %H:%M')})

# PARTS
for _, r in parts.iterrows():
    pid = r['Part_ID']
    if pd.isna(r.get('Part_Name')): log('Parts',pid,'Part_Name','Null Value','Part name missing')
    if pd.isna(r.get('Revision')): log('Parts',pid,'Revision','Null Value','Revision missing')
    if pd.notna(r.get('Unit_Cost_INR')) and r['Unit_Cost_INR']<0: log('Parts',pid,'Unit_Cost_INR','Invalid Value',f'Negative: {r["Unit_Cost_INR"]}')
    if pd.isna(r.get('Weight_KG')): log('Parts',pid,'Weight_KG','Null Value','Weight missing','Medium')
    if pd.notna(r.get('Status')) and r['Status'] not in VALID_STATUSES: log('Parts',pid,'Status','Invalid Reference',f'Unknown: {r["Status"]}')

# BOM
for _, r in bom.iterrows():
    bid = r['BOM_ID']
    if pd.isna(r.get('Parent_Part_ID')): log('BOM',bid,'Parent_Part_ID','Null Value','Parent ID missing')
    elif r['Parent_Part_ID'] not in valid_part_ids: log('BOM',bid,'Parent_Part_ID','Orphan Record',f'Not in Parts: {r["Parent_Part_ID"]}')
    if pd.notna(r.get('Child_Part_ID')) and r['Child_Part_ID'] not in valid_part_ids: log('BOM',bid,'Child_Part_ID','Orphan Record',f'Not in Parts: {r["Child_Part_ID"]}')
    if pd.notna(r.get('Quantity')) and r['Quantity']<=0: log('BOM',bid,'Quantity','Invalid Value',f'Qty<=0: {r["Quantity"]}')

# DOCUMENTS
for _, r in docs.iterrows():
    did = r['Doc_ID']
    if pd.isna(r.get('Doc_Name')): log('Documents',did,'Doc_Name','Null Value','Doc name missing')
    if pd.notna(r.get('Related_Part_ID')) and r['Related_Part_ID'] not in valid_part_ids: log('Documents',did,'Related_Part_ID','Orphan Record',f'Part not found: {r["Related_Part_ID"]}')

# ECO
for _, r in eco.iterrows():
    eid = r['ECO_ID']
    if pd.isna(r.get('Priority')): log('ECO',eid,'Priority','Null Value','Priority missing')
    if pd.notna(r.get('Status')) and r['Status'] not in VALID_ECO_STS: log('ECO',eid,'Status','Invalid Reference',f'Unknown: {r["Status"]}')
    if r.get('Status')=='Approved' and pd.isna(r.get('Approved_By')): log('ECO',eid,'Approved_By','Business Rule Violation','Approved ECO has no approver','Critical')

issues_df = pd.DataFrame(issues)
print(f'🔴 Total Issues Found: {len(issues_df)}')
print(issues_df.groupby(['Entity','Severity']).size().rename('Count').to_string())

🔴 Total Issues Found: 18
Entity     Severity
BOM        High        3
Documents  High        2
ECO        Critical    6
           High        2
Parts      High        4
           Medium      1


In [6]:
issues_df

,Entity,Record_ID,Field,Issue_Type,Detail,Severity,Detected_At
0,Parts,PT-00006,Part_Name,Null Value,Part name missing,High,2026-04-06 17:15
1,Parts,PT-00013,Unit_Cost_INR,Invalid Value,Negative: -999.0,High,2026-04-06 17:15
2,Parts,PT-00026,Revision,Null Value,Revision missing,High,2026-04-06 17:15
3,Parts,PT-00041,Status,Invalid Reference,Unknown: INVALID_STATUS,High,2026-04-06 17:15
4,Parts,PT-00061,Weight_KG,Null Value,Weight missing,Medium,2026-04-06 17:15
5,BOM,BOM-00004,Quantity,Invalid Value,Qty<=0: 0,High,2026-04-06 17:15
6,BOM,BOM-00011,Parent_Part_ID,Null Value,Parent ID missing,High,2026-04-06 17:15
7,BOM,BOM-00023,Child_Part_ID,Orphan Record,Not in Parts: PT-99999,High,2026-04-06 17:15
8,Documents,DOC-00008,Doc_Name,Null Value,Doc name missing,High,2026-04-06 17:15
9,Documents,DOC-00016,Related_Part_ID,Orphan Record,Part not found: PT-00000,High,2026-04-06 17:15


---
## 🧪 Test Case Execution (UAT / SDLC Aligned)

In [7]:
results = pd.read_csv('test_results.csv')
print(f'Total TCs : {len(results)}')
print(f'Passed    : {(results["Status"]=="PASS").sum()}')
print(f'Failed    : {(results["Status"]=="FAIL").sum()}')
print(f'Pass Rate : {(results["Status"]=="PASS").mean()*100:.1f}%')

Total TCs : 31
Passed    : 31
Failed    : 0
Pass Rate : 100.0%


In [8]:
print('Results by Category:')
print(results.groupby(['Category','Status']).size().unstack(fill_value=0).to_string())

Results by Category:
Status          PASS
Category            
Data Integrity     6
Extraction         8
Migration QA       4
Regression         4
Validation         9


In [9]:
results[['Test_Case_ID','Test_Name','Entity','Category','Status','Severity','Remarks']]

,Test_Case_ID,Test_Name,Entity,Category,Status,Severity,Remarks
0,TC-001,Parts Record Count,Parts,Extraction,PASS,High,NaN
1,TC-002,BOM Record Count,BOM,Extraction,PASS,High,NaN
2,TC-003,Document Record Count,Docs,Extraction,PASS,High,NaN
3,TC-004,ECO Record Count,ECO,Extraction,PASS,High,NaN
4,TC-005,Parts Column Count,Parts,Extraction,PASS,Medium,NaN
5,TC-006,BOM Column Count,BOM,Extraction,PASS,Medium,NaN
6,TC-007,Source System Tag - Parts,Parts,Extraction,PASS,Medium,NaN
7,TC-008,Source System Tag - BOM,BOM,Extraction,PASS,Medium,NaN
8,TC-009,Parts - Unique IDs,Parts,Data Integrity,PASS,Critical,NaN
9,TC-010,BOM - Unique IDs,BOM,Data Integrity,PASS,Critical,NaN


---
## 📊  Migration Status Summary

In [10]:
print('Migration Status Summary:')
for name, df in [('Parts',parts),('BOM',bom),('Documents',docs),('ECO',eco)]:
    total = len(df)
    comp  = (df['Migration_Status']=='Completed').sum()
    fail  = (df['Migration_Status']=='Failed').sum()
    pend  = (df['Migration_Status']=='Pending').sum()
    print(f'\n  {name}:')
    print(f'    Completed : {comp} ({comp/total*100:.1f}%)')
    print(f'    Failed    : {fail} ({fail/total*100:.1f}%)')
    print(f'    Pending   : {pend} ({pend/total*100:.1f}%)')

Migration Status Summary:

  Parts:
    Completed : 87 (43.5%)
    Failed    : 17 (8.5%)
    Pending   : 51 (25.5%)

  BOM:
    Completed : 86 (57.3%)
    Failed    : 23 (15.3%)
    Pending   : 41 (27.3%)

  Documents:
    Completed : 52 (52.0%)
    Failed    : 13 (13.0%)
    Pending   : 35 (35.0%)

  ECO:
    Completed : 48 (60.0%)
    Failed    : 10 (12.5%)
    Pending   : 22 (27.5%)


---
##  Key Findings & Recommendations

In [11]:
print('='*60)
print('PLM MIGRATION — KEY FINDINGS & RECOMMENDATIONS')
print('='*60)
print(f'\n📌 Total Issues Detected : {len(issues_df)}')
crit = (issues_df['Severity']=='Critical').sum()
print(f'   Critical             : {crit}  ← Must fix before go-live')
print(f'   High                 : {(issues_df["Severity"]=="High").sum()}')
print(f'   Medium               : {(issues_df["Severity"]=="Medium").sum()}')
print(f'\n🧪 Test Execution        : {len(results)} test cases')
print(f'   Pass Rate            : {(results["Status"]=="PASS").mean()*100:.1f}%')
print('\n✅ RECOMMENDATIONS')
print('   1. Resolve all Critical issues (Approved ECOs without approver) before migration')
print('   2. Cleanse null Part Names and invalid Status values in source system')
print('   3. Fix BOM orphan records — Child Part IDs must exist in Parts master')
print('   4. Run UAT sign-off after all High severity issues are resolved')
print('   5. Execute post-migration count check SQL to verify record parity')
print('\n' + '='*60)
print('End of Migration Simulation Report')
print('='*60)

PLM MIGRATION — KEY FINDINGS & RECOMMENDATIONS

📌 Total Issues Detected : 18
   Critical             : 6  ← Must fix before go-live
   High                 : 11
   Medium               : 1

🧪 Test Execution        : 31 test cases
   Pass Rate            : 100.0%

✅ RECOMMENDATIONS
   1. Resolve all Critical issues (Approved ECOs without approver) before migration
   2. Cleanse null Part Names and invalid Status values in source system
   3. Fix BOM orphan records — Child Part IDs must exist in Parts master
   4. Run UAT sign-off after all High severity issues are resolved
   5. Execute post-migration count check SQL to verify record parity

End of Migration Simulation Report
